# FOMO Object Detection for ESP32-CAM

**Faster Objects, More Objects (FOMO)** — a lightweight centroid-based object detector
designed for microcontrollers.

## Architecture Overview

Unlike traditional detectors (YOLO, SSD) that predict bounding boxes, FOMO:
1. Divides the image into a grid (e.g., 12×12 for 96×96 input)
2. Each grid cell predicts **which class** is centered there (or background)
3. No bounding box regression → drastically fewer parameters
4. Uses a truncated MobileNetV2 backbone (stride-8) for features

This makes it ideal for ESP32-CAM with ~4MB PSRAM and ~520KB SRAM.

### Target Classes
| Index | Class | COCO ID | Purpose |
|-------|-------|---------|----------|
| 0 | background | — | No object |
| 1 | person | 1 | Interaction target |
| 2 | sports_ball | 37 | Fetch/track target |
| 3 | chair | 62 | Obstacle avoidance |
| 4 | couch | 63 | Obstacle avoidance |
| 5 | dining_table | 67 | Obstacle avoidance |

### Memory Budget
- **Model size target**: < 250 KB (int8 quantized TFLite)
- **Input tensor**: 96×96×3 = 27,648 bytes
- **Output tensor**: 12×12×6 = 864 bytes
- **Arena (TFLite Micro)**: ~300 KB (fits in PSRAM)

## 1. Environment Setup

In [ ]:
# Install dependencies
# !pip install torch torchvision albumentations pycocotools matplotlib numpy onnx ai-edge-torch
# For COCO dataset download:
# !pip install fiftyone  # (optional, or download manually)

In [2]:
import os
import json
import math
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

import albumentations as A
from albumentations.pytorch import ToTensorV2
from pycocotools.coco import COCO

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cpu


## 2. Configuration

In [ ]:
# ── Model / training config ──────────────────────────────────────────────
INPUT_SIZE     = 96        # 96x96 pixels — fits ESP32 memory budget
GRID_SIZE      = 12        # INPUT_SIZE / stride(8) = 12
CELL_SIZE      = INPUT_SIZE // GRID_SIZE  # 8 pixels per cell
NUM_CLASSES    = 6         # background + 5 object classes
BATCH_SIZE     = 64
LEARNING_RATE  = 1e-3
NUM_EPOCHS     = 30
WEIGHT_DECAY   = 1e-4

# ── COCO class mapping ───────────────────────────────────────────────────
# Maps COCO category IDs → our internal class indices (1-indexed, 0=background)
COCO_TO_FOMO = {
    1:  1,   # person
    37: 2,   # sports_ball
    62: 3,   # chair
    63: 4,   # couch
    67: 5,   # dining_table
}
FOMO_CLASS_NAMES = ['background', 'person', 'sports_ball', 'chair', 'couch', 'dining_table']
TARGET_COCO_IDS = list(COCO_TO_FOMO.keys())

# ── Dataset paths ────────────────────────────────────────────────────────
COCO_ROOT       = Path('./datasets/coco')
COCO_TRAIN_IMGS = COCO_ROOT / 'train2017'
COCO_VAL_IMGS   = COCO_ROOT / 'val2017'
COCO_TRAIN_ANN  = COCO_ROOT / 'annotations' / 'instances_train2017.json'
COCO_VAL_ANN    = COCO_ROOT / 'annotations' / 'instances_val2017.json'

# ── Output paths ─────────────────────────────────────────────────────────
OUTPUT_DIR = Path('./output')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Grid: {GRID_SIZE}x{GRID_SIZE} ({GRID_SIZE**2} cells)')
print(f'Each cell covers {CELL_SIZE}x{CELL_SIZE} pixels')
print(f'Classes: {FOMO_CLASS_NAMES}')

## 3. Download & Filter COCO Dataset

We only need images that contain at least one of our target classes.
This reduces the dataset from ~118K to ~30K images.

In [ ]:
%%bash
# Download COCO 2017 if not present (this is ~20 GB total)
# Uncomment the lines below if you need to download

mkdir -p coco && cd coco

# Train images
# wget -q http://images.cocodataset.org/zips/train2017.zip && unzip -q train2017.zip

# Val images
# wget -q http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip

# Annotations
# wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q annotations_trainval2017.zip

echo "COCO directory contents:"
ls -la

In [ ]:
def load_filtered_coco(annotation_path, target_cat_ids):
    """
    Load COCO annotations, keeping only images that contain
    at least one annotation from our target classes.
    
    Returns:
        coco: COCO API object
        image_ids: list of filtered image IDs
    """
    coco = COCO(str(annotation_path))
    
    # Collect image IDs that have at least one target annotation
    image_ids = set()
    for cat_id in target_cat_ids:
        image_ids.update(coco.getImgIds(catIds=[cat_id]))
    image_ids = sorted(image_ids)
    
    # Count per class
    class_counts = defaultdict(int)
    for img_id in image_ids:
        ann_ids = coco.getAnnIds(imgIds=img_id, catIds=target_cat_ids, iscrowd=False)
        for ann in coco.loadAnns(ann_ids):
            if ann['category_id'] in target_cat_ids:
                class_counts[ann['category_id']] += 1
    
    print(f'Filtered to {len(image_ids)} images')
    for cat_id, count in sorted(class_counts.items()):
        cat_name = coco.loadCats(cat_id)[0]['name']
        print(f'  {cat_name} (id={cat_id}): {count} annotations')
    
    return coco, image_ids

print('Loading training set...')
coco_train, train_ids = load_filtered_coco(COCO_TRAIN_ANN, TARGET_COCO_IDS)

print('\nLoading validation set...')
coco_val, val_ids = load_filtered_coco(COCO_VAL_ANN, TARGET_COCO_IDS)

## 4. Data Augmentation Pipeline

The OV2640 on ESP32-CAM produces noisy, low-dynamic-range images.
We simulate this during training so the model is robust to real-world input.

**Key augmentations**:
- `GaussNoise(var_limit=(100,500))` — simulates sensor noise
- `Blur(blur_limit=3)` — simulates motion blur / out-of-focus
- `RandomBrightnessContrast` — accounts for varying lighting
- `HueSaturationValue` — handles color cast from the sensor

In [ ]:
# ── Training augmentations (simulate ESP32-CAM sensor degradation) ────────
train_transform = A.Compose([
    A.Resize(INPUT_SIZE, INPUT_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.GaussNoise(var_limit=(100, 500), p=0.4),  # Crucial for ESP32 sensors
    A.Blur(blur_limit=3, p=0.3),                # Motion / focus blur
    A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=20,
        val_shift_limit=20,
        p=0.3
    ),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], bbox_params=A.BboxParams(
    format='coco',                   # [x_min, y_min, width, height]
    label_fields=['category_ids'],
    min_area=16,                     # Drop tiny boxes post-resize
    min_visibility=0.2,
))

# ── Validation transform (no augmentation, just resize + normalize) ───────
val_transform = A.Compose([
    A.Resize(INPUT_SIZE, INPUT_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], bbox_params=A.BboxParams(
    format='coco',
    label_fields=['category_ids'],
    min_area=16,
    min_visibility=0.2,
))

print('Augmentation pipelines ready.')

## 5. FOMO Dataset: Bounding Boxes → Centroid Grid

The key insight of FOMO: instead of predicting bounding boxes, we convert
each annotation into a **centroid on a grid**.

```
Image (96x96)          Grid (12x12)
┌──────────────┐       ┌─┬─┬─┬─┬─┬─┬─┬─┬─┬─┬─┬─┐
│              │       │0│0│0│0│0│0│0│0│0│0│0│0│
│   ┌──┐       │       ├─┼─┼─┼─┼─┼─┼─┼─┼─┼─┼─┼─┤
│   │🏀│       │  →    │0│0│2│0│0│0│0│0│0│0│0│0│  ← ball centroid
│   └──┘       │       ├─┼─┼─┼─┼─┼─┼─┼─┼─┼─┼─┼─┤
│        ┌───┐ │       │0│0│0│0│0│0│0│0│0│0│0│0│
│        │🧑│ │       ├─┼─┼─┼─┼─┼─┼─┼─┼─┼─┼─┼─┤
│        └───┘ │       │0│0│0│0│0│0│0│1│0│0│0│0│  ← person centroid
└──────────────┘       └─┴─┴─┴─┴─┴─┴─┴─┴─┴─┴─┴─┘
```

Each grid cell = 8×8 pixels. If multiple objects land in the same cell,
the last one wins (rare at this resolution).

In [ ]:
class FOMODataset(Dataset):
    """
    Converts COCO annotations into FOMO centroid-grid targets.
    
    For each image:
      - Loads image + bboxes for our target classes
      - Applies augmentations (which also transform bboxes)
      - Converts each bbox centroid to a grid cell assignment
      - Returns image tensor [3, 96, 96] and target grid [12, 12]
    """
    
    def __init__(self, coco, image_ids, image_dir, transform,
                 coco_to_fomo=COCO_TO_FOMO, input_size=INPUT_SIZE,
                 grid_size=GRID_SIZE):
        self.coco = coco
        self.image_ids = image_ids
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.coco_to_fomo = coco_to_fomo
        self.target_cat_ids = list(coco_to_fomo.keys())
        self.input_size = input_size
        self.grid_size = grid_size
        self.cell_size = input_size / grid_size
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = self.image_dir / img_info['file_name']
        
        # Load image
        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f'Image not found: {img_path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Load annotations for this image (only our target classes)
        ann_ids = self.coco.getAnnIds(
            imgIds=img_id,
            catIds=self.target_cat_ids,
            iscrowd=False
        )
        anns = self.coco.loadAnns(ann_ids)
        
        # Extract bboxes and category IDs
        bboxes = []
        category_ids = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            if w > 0 and h > 0:
                bboxes.append([x, y, w, h])
                category_ids.append(ann['category_id'])
        
        # Apply augmentations
        transformed = self.transform(
            image=image,
            bboxes=bboxes,
            category_ids=category_ids
        )
        
        image_tensor = transformed['image']  # [3, H, W] float32
        aug_bboxes = transformed['bboxes']   # post-augmentation bboxes
        aug_cats = transformed['category_ids']
        
        # ── Build centroid grid target ────────────────────────────────────
        # Shape: [grid_size, grid_size], dtype=long
        # Values: 0=background, 1-5=object class
        target = torch.zeros(self.grid_size, self.grid_size, dtype=torch.long)
        
        for bbox, cat_id in zip(aug_bboxes, aug_cats):
            x, y, w, h = bbox
            # Centroid in pixel coordinates
            cx = x + w / 2.0
            cy = y + h / 2.0
            
            # Map to grid cell
            gx = int(cx / self.cell_size)
            gy = int(cy / self.cell_size)
            
            # Clamp to grid bounds
            gx = max(0, min(gx, self.grid_size - 1))
            gy = max(0, min(gy, self.grid_size - 1))
            
            # Map COCO category to FOMO class index
            fomo_class = self.coco_to_fomo.get(cat_id, 0)
            if fomo_class > 0:
                target[gy, gx] = fomo_class
        
        return image_tensor, target


# ── Create datasets ──────────────────────────────────────────────────────
train_dataset = FOMODataset(
    coco_train, train_ids, COCO_TRAIN_IMGS, train_transform
)
val_dataset = FOMODataset(
    coco_val, val_ids, COCO_VAL_IMGS, val_transform
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

print(f'Training samples:   {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Training batches:   {len(train_loader)}')

In [ ]:
# ── Visualize a few training samples ─────────────────────────────────────

def denormalize(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """Reverse ImageNet normalization for display."""
    img = tensor.clone()
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    return img.clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, ax in enumerate(axes.flat):
    img, target = train_dataset[i]
    img_display = denormalize(img).permute(1, 2, 0).numpy()
    
    ax.imshow(img_display)
    
    # Overlay grid cells with detections
    for gy in range(GRID_SIZE):
        for gx in range(GRID_SIZE):
            cls = target[gy, gx].item()
            if cls > 0:
                # Draw a marker at the cell center
                px = (gx + 0.5) * CELL_SIZE
                py = (gy + 0.5) * CELL_SIZE
                colors = ['', 'red', 'yellow', 'cyan', 'magenta', 'green']
                ax.plot(px, py, 'o', color=colors[cls], markersize=8, markeredgewidth=2)
                ax.text(px + 3, py - 3, FOMO_CLASS_NAMES[cls],
                        fontsize=6, color=colors[cls],
                        bbox=dict(boxstyle='round,pad=0.1', facecolor='black', alpha=0.7))
    
    obj_count = (target > 0).sum().item()
    ax.set_title(f'Sample {i} — {obj_count} objects')
    ax.axis('off')

plt.suptitle('Training Samples with FOMO Grid Centroids', fontsize=14)
plt.tight_layout()
plt.show()

## 6. FOMO Model Architecture

### Didactic: Building FOMO from Scratch

FOMO's architecture is elegant in its simplicity:

```
Input [B, 3, 96, 96]
       │
       ▼
┌──────────────────┐
│  MobileNetV2     │   Pretrained backbone, truncated at stride-8
│  features[:7]    │   Output: [B, 32, 12, 12]
└──────────────────┘
       │
       ▼
┌──────────────────┐
│  1×1 Conv (32→16)│   Reduce channels
│  BatchNorm + ReLU│
└──────────────────┘
       │
       ▼
┌──────────────────┐
│  1×1 Conv (16→6) │   Predict class per cell
└──────────────────┘
       │
       ▼
Output [B, 6, 12, 12]   Per-cell class logits
```

**Why MobileNetV2 features[:7]?**

MobileNetV2 has 19 feature blocks (features[0..18]). For a 96×96 input:

| Layer Range | Output Shape | Stride | Notes |
|-------------|-------------|--------|-------|
| features[0] | 48×48×32 | 2 | Initial conv |
| features[1] | 48×48×16 | 2 | First bottleneck |
| features[2:4] | 24×24×24 | 4 | |
| features[4:7] | **12×12×32** | **8** | ← We cut here |
| features[7:14] | 6×6×96 | 16 | Too low resolution |
| features[14:18] | 3×3×320 | 32 | Way too small |

Cutting at stride 8 gives us a 12×12 grid — each cell covers 8×8 pixels.
This is the sweet spot: enough resolution for centroid detection,
small enough for ESP32.

In [ ]:
class FOMOModel(nn.Module):
    """
    FOMO: Faster Objects, More Objects
    
    A centroid-based object detector designed for microcontrollers.
    Uses a truncated MobileNetV2 backbone with a lightweight classification head.
    
    Args:
        num_classes: Total classes INCLUDING background (default=6)
        backbone_cutoff: Where to truncate MobileNetV2 features (default=7)
        dropout: Dropout rate before final conv (default=0.1)
    """
    
    def __init__(self, num_classes=NUM_CLASSES, backbone_cutoff=7, dropout=0.1):
        super().__init__()
        
        # Load pretrained MobileNetV2 and truncate
        full_backbone = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(full_backbone.features[:backbone_cutoff]))
        
        # Determine backbone output channels by doing a dummy forward pass
        with torch.no_grad():
            dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
            backbone_out = self.backbone(dummy)
            backbone_channels = backbone_out.shape[1]
            spatial = backbone_out.shape[2]
        print(f'Backbone output: {backbone_channels}ch × {spatial}×{spatial}')
        
        # Classification head: two 1×1 convolutions
        self.head = nn.Sequential(
            nn.Conv2d(backbone_channels, 16, kernel_size=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU6(inplace=True),
            nn.Dropout2d(p=dropout),
            nn.Conv2d(16, num_classes, kernel_size=1),
        )
        
        # Freeze early backbone layers (first 3 blocks) for faster convergence
        for i, layer in enumerate(self.backbone):
            if i < 3:
                for param in layer.parameters():
                    param.requires_grad = False
    
    def forward(self, x):
        """Forward pass.
        
        Args:
            x: [B, 3, 96, 96] input images (normalized)
        
        Returns:
            logits: [B, num_classes, 12, 12] per-cell class logits
        """
        features = self.backbone(x)
        logits = self.head(features)
        return logits
    
    def predict(self, x, conf_threshold=0.5):
        """Run inference and return detections.
        
        Returns list of (class_id, grid_x, grid_y, confidence) tuples.
        """
        self.eval()
        with torch.no_grad():
            logits = self.forward(x)                # [B, C, H, W]
            probs = F.softmax(logits, dim=1)        # [B, C, H, W]
            
            detections = []
            for b in range(x.shape[0]):
                batch_dets = []
                conf, pred_class = probs[b].max(dim=0)  # [H, W] each
                
                for gy in range(pred_class.shape[0]):
                    for gx in range(pred_class.shape[1]):
                        cls = pred_class[gy, gx].item()
                        c = conf[gy, gx].item()
                        if cls > 0 and c >= conf_threshold:
                            batch_dets.append((cls, gx, gy, c))
                detections.append(batch_dets)
            
            return detections


# ── Instantiate ──────────────────────────────────────────────────────────
model = FOMOModel(num_classes=NUM_CLASSES).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Frozen parameters:    {total_params - trainable_params:,}')

## 7. Training Loop

### Loss Function

FOMO uses **weighted cross-entropy** per grid cell. The weighting is critical
because ~95% of cells are background — without it, the model would just
predict background everywhere.

We compute class weights inversely proportional to frequency, then apply
a focal-loss-inspired modulation to further focus on hard examples.

In [ ]:
def compute_class_weights(dataloader, num_classes, num_batches=50):
    """
    Sample a subset of the training data to estimate class frequency.
    Returns inverse-frequency weights for cross-entropy.
    """
    counts = torch.zeros(num_classes)
    total_cells = 0
    
    for i, (_, targets) in enumerate(dataloader):
        if i >= num_batches:
            break
        for cls in range(num_classes):
            counts[cls] += (targets == cls).sum().item()
        total_cells += targets.numel()
    
    # Inverse frequency weighting with smoothing
    freq = counts / total_cells
    weights = 1.0 / (freq + 1e-6)
    weights = weights / weights.sum() * num_classes  # normalize
    
    print('Class frequencies:')
    for i, (f, w) in enumerate(zip(freq, weights)):
        print(f'  {FOMO_CLASS_NAMES[i]:15s}: freq={f:.4f}  weight={w:.2f}')
    
    return weights

class_weights = compute_class_weights(train_loader, NUM_CLASSES)
class_weights = class_weights.to(DEVICE)

In [ ]:
# ── Optimizer & Scheduler ────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Cosine annealing with warm restarts
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Weighted cross-entropy loss
criterion = nn.CrossEntropyLoss(weight=class_weights)

print('Optimizer: AdamW')
print(f'Initial LR: {LEARNING_RATE}')
print(f'Scheduler: CosineAnnealingWarmRestarts (T_0=10)')

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    obj_correct = 0
    obj_total = 0
    
    for images, targets in loader:
        images = images.to(device)     # [B, 3, 96, 96]
        targets = targets.to(device)   # [B, 12, 12]
        
        logits = model(images)         # [B, 6, 12, 12]
        
        # CrossEntropyLoss expects [B, C, H, W] logits and [B, H, W] targets
        loss = criterion(logits, targets)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
        # Accuracy stats
        preds = logits.argmax(dim=1)   # [B, 12, 12]
        correct += (preds == targets).sum().item()
        total += targets.numel()
        
        # Object-only accuracy (non-background cells)
        obj_mask = targets > 0
        if obj_mask.any():
            obj_correct += (preds[obj_mask] == targets[obj_mask]).sum().item()
            obj_total += obj_mask.sum().item()
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / total
    epoch_obj_acc = obj_correct / max(obj_total, 1)
    
    return epoch_loss, epoch_acc, epoch_obj_acc


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    obj_correct = 0
    obj_total = 0
    
    # Per-class tracking
    class_tp = torch.zeros(NUM_CLASSES)
    class_fp = torch.zeros(NUM_CLASSES)
    class_fn = torch.zeros(NUM_CLASSES)
    
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        
        logits = model(images)
        loss = criterion(logits, targets)
        running_loss += loss.item() * images.size(0)
        
        preds = logits.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.numel()
        
        obj_mask = targets > 0
        if obj_mask.any():
            obj_correct += (preds[obj_mask] == targets[obj_mask]).sum().item()
            obj_total += obj_mask.sum().item()
        
        # Per-class precision/recall
        for cls in range(1, NUM_CLASSES):
            pred_cls = (preds == cls)
            true_cls = (targets == cls)
            class_tp[cls] += (pred_cls & true_cls).sum().item()
            class_fp[cls] += (pred_cls & ~true_cls).sum().item()
            class_fn[cls] += (~pred_cls & true_cls).sum().item()
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / total
    epoch_obj_acc = obj_correct / max(obj_total, 1)
    
    # F1 per class
    class_metrics = {}
    for cls in range(1, NUM_CLASSES):
        precision = class_tp[cls] / max(class_tp[cls] + class_fp[cls], 1)
        recall = class_tp[cls] / max(class_tp[cls] + class_fn[cls], 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-6)
        class_metrics[FOMO_CLASS_NAMES[cls]] = {
            'precision': precision.item(),
            'recall': recall.item(),
            'f1': f1.item()
        }
    
    return epoch_loss, epoch_acc, epoch_obj_acc, class_metrics

print('Training and validation functions ready.')

In [ ]:
# ── Main training loop ───────────────────────────────────────────────────

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
           'train_obj_acc': [], 'val_obj_acc': [], 'lr': []}

for epoch in range(1, NUM_EPOCHS + 1):
    current_lr = optimizer.param_groups[0]['lr']
    
    # Train
    train_loss, train_acc, train_obj_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE
    )
    
    # Validate
    val_loss, val_acc, val_obj_acc, class_metrics = validate(
        model, val_loader, criterion, DEVICE
    )
    
    scheduler.step()
    
    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_obj_acc'].append(train_obj_acc)
    history['val_obj_acc'].append(val_obj_acc)
    history['lr'].append(current_lr)
    
    print(f'Epoch {epoch:2d}/{NUM_EPOCHS}  '
          f'lr={current_lr:.6f}  '
          f'train_loss={train_loss:.4f}  '
          f'val_loss={val_loss:.4f}  '
          f'obj_acc={val_obj_acc:.3f}')
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': val_loss,
            'val_obj_acc': val_obj_acc,
            'class_metrics': class_metrics,
        }, OUTPUT_DIR / 'fomo_best.pth')
        print(f'  ✓ Saved best model (val_loss={val_loss:.4f})')
    
    # Print per-class metrics every 5 epochs
    if epoch % 5 == 0:
        print('  Per-class metrics:')
        for cls_name, m in class_metrics.items():
            print(f'    {cls_name:15s}: P={m["precision"]:.3f}  R={m["recall"]:.3f}  F1={m["f1"]:.3f}')

print('\nTraining complete!')

In [ ]:
# ── Plot training curves ─────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Object accuracy
axes[1].plot(history['train_obj_acc'], label='Train obj acc')
axes[1].plot(history['val_obj_acc'], label='Val obj acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Object Cell Accuracy (non-background)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(history['lr'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('LR Schedule')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

## 8. Evaluation & Visualization

In [ ]:
# ── Load best model and visualize predictions ────────────────────────────

checkpoint = torch.load(OUTPUT_DIR / 'fomo_best.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]}')
print(f'Val loss: {checkpoint["val_loss"]:.4f}')
print(f'Val object accuracy: {checkpoint["val_obj_acc"]:.3f}')
print('\nPer-class metrics:')
for cls_name, m in checkpoint['class_metrics'].items():
    print(f'  {cls_name:15s}: P={m["precision"]:.3f}  R={m["recall"]:.3f}  F1={m["f1"]:.3f}')

In [ ]:
# ── Visualize predictions on validation set ──────────────────────────────

model.eval()
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
colors = {1: 'red', 2: 'yellow', 3: 'cyan', 4: 'magenta', 5: 'green'}

for i, ax in enumerate(axes.flat):
    img, target = val_dataset[i * 10]  # Sample every 10th image
    img_tensor = img.unsqueeze(0).to(DEVICE)
    
    # Get predictions
    detections = model.predict(img_tensor, conf_threshold=0.4)[0]
    
    # Display
    img_display = denormalize(img).permute(1, 2, 0).numpy()
    ax.imshow(img_display)
    
    # Draw ground truth (circles)
    for gy in range(GRID_SIZE):
        for gx in range(GRID_SIZE):
            cls = target[gy, gx].item()
            if cls > 0:
                px = (gx + 0.5) * CELL_SIZE
                py = (gy + 0.5) * CELL_SIZE
                ax.plot(px, py, 'o', color=colors[cls], markersize=12,
                        markeredgewidth=2, fillstyle='none', label='GT')
    
    # Draw predictions (crosses)
    for cls, gx, gy, conf in detections:
        px = (gx + 0.5) * CELL_SIZE
        py = (gy + 0.5) * CELL_SIZE
        ax.plot(px, py, 'x', color=colors[cls], markersize=10, markeredgewidth=2)
        ax.text(px + 3, py - 3, f'{FOMO_CLASS_NAMES[cls]} {conf:.0%}',
                fontsize=6, color='white',
                bbox=dict(facecolor=colors[cls], alpha=0.7, pad=1))
    
    n_gt = (target > 0).sum().item()
    n_pred = len(detections)
    ax.set_title(f'GT={n_gt}  Pred={n_pred}', fontsize=10)
    ax.axis('off')

# Legend
legend_elements = [
    mpatches.Patch(facecolor=c, label=FOMO_CLASS_NAMES[i])
    for i, c in colors.items()
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5, fontsize=10)

plt.suptitle('Predictions (×) vs Ground Truth (○)', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'val_predictions.png', dpi=150)
plt.show()

## 9. Export to TFLite (Int8 Quantized)

The export pipeline:
1. **PyTorch → ONNX** — standard export
2. **ONNX → TFLite** — using `ai-edge-torch` (Google's tool) or `onnx-tf` + TFLite converter
3. **Post-training quantization** — int8 weights + activations using representative dataset

Target: model file < 250 KB for ESP32 flash.

In [ ]:
# ── Step 1: Export to ONNX ───────────────────────────────────────────────

model.eval()
model_cpu = model.to('cpu')

dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
onnx_path = OUTPUT_DIR / 'fomo_model.onnx'

torch.onnx.export(
    model_cpu,
    dummy_input,
    str(onnx_path),
    opset_version=13,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes=None,  # Fixed batch size of 1 for MCU
)

onnx_size = onnx_path.stat().st_size / 1024
print(f'ONNX model saved: {onnx_path} ({onnx_size:.1f} KB)')

In [ ]:
# ── Step 2: ONNX → TFLite with int8 quantization ────────────────────────
#
# Option A: Using ai-edge-torch (recommended, newer)
# Option B: Using onnx-tf + TFLite converter (fallback)
#
# We show both options — use whichever works in your environment.

# === Option A: ai-edge-torch ===
try:
    import ai_edge_torch
    
    # Representative dataset for calibration
    def representative_dataset():
        """Yields calibration samples for quantization."""
        for i in range(100):
            img, _ = val_dataset[i]
            yield img.unsqueeze(0).numpy()
    
    # Convert with int8 quantization
    edge_model = ai_edge_torch.convert(
        model_cpu,
        (dummy_input,),
        quant_config=ai_edge_torch.quantize.pt2e_quantizer.get_symmetric_quantization_config(
            is_per_channel=True,
            is_dynamic=False,
        ),
    )
    
    tflite_path = OUTPUT_DIR / 'fomo_model_int8.tflite'
    edge_model.export(str(tflite_path))
    print(f'[ai-edge-torch] TFLite model saved: {tflite_path}')
    
except ImportError:
    print('ai-edge-torch not available, falling back to Option B...')
    
    # === Option B: onnx-tf + TFLite converter ===
    import onnx
    from onnx_tf.backend import prepare
    import tensorflow as tf
    
    # ONNX → TensorFlow SavedModel
    onnx_model = onnx.load(str(onnx_path))
    tf_rep = prepare(onnx_model)
    saved_model_dir = str(OUTPUT_DIR / 'fomo_saved_model')
    tf_rep.export_graph(saved_model_dir)
    print(f'TF SavedModel exported to: {saved_model_dir}')
    
    # SavedModel → TFLite with int8 quantization
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    
    # Representative dataset for full int8 quantization
    def representative_dataset_gen():
        for i in range(100):
            img, _ = val_dataset[i]
            sample = img.unsqueeze(0).numpy().astype(np.float32)
            yield [sample]
    
    converter.representative_dataset = representative_dataset_gen
    
    tflite_model = converter.convert()
    
    tflite_path = OUTPUT_DIR / 'fomo_model_int8.tflite'
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    
    print(f'[onnx-tf] TFLite model saved: {tflite_path}')

# Report size
tflite_size = tflite_path.stat().st_size / 1024
print(f'\nTFLite model size: {tflite_size:.1f} KB')
if tflite_size < 250:
    print('✓ Fits in ESP32 flash budget (<250 KB)')
else:
    print('⚠ Model exceeds 250 KB — consider reducing backbone_cutoff or channels')

In [ ]:
# ── Step 3: Verify the TFLite model ──────────────────────────────────────

import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('Input details:')
for d in input_details:
    print(f'  shape={d["shape"]}  dtype={d["dtype"]}  '
          f'quant={d.get("quantization_parameters", {})}')

print('\nOutput details:')
for d in output_details:
    print(f'  shape={d["shape"]}  dtype={d["dtype"]}  '
          f'quant={d.get("quantization_parameters", {})}')

In [ ]:
# ── Test TFLite model accuracy ───────────────────────────────────────────

def run_tflite_inference(interpreter, image_np):
    """Run one inference on the TFLite model."""
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    
    # Quantize input if needed
    if input_details['dtype'] == np.int8:
        quant = input_details['quantization_parameters']
        scale = quant['scales'][0]
        zero_point = quant['zero_points'][0]
        image_quantized = (image_np / scale + zero_point).astype(np.int8)
        interpreter.set_tensor(input_details['index'], image_quantized)
    else:
        interpreter.set_tensor(input_details['index'], image_np.astype(np.float32))
    
    interpreter.invoke()
    output = interpreter.get_tensor(output_details['index'])
    
    # Dequantize output if needed
    if output_details['dtype'] == np.int8:
        quant = output_details['quantization_parameters']
        scale = quant['scales'][0]
        zero_point = quant['zero_points'][0]
        output = (output.astype(np.float32) - zero_point) * scale
    
    return output


# Compare PyTorch vs TFLite on 50 validation samples
model_cpu.eval()
match_count = 0
total_cells = 0

for i in range(50):
    img, target = val_dataset[i]
    img_np = img.unsqueeze(0).numpy()
    
    # PyTorch prediction
    with torch.no_grad():
        pt_logits = model_cpu(img.unsqueeze(0))
        pt_pred = pt_logits.argmax(dim=1).squeeze().numpy()
    
    # TFLite prediction
    tfl_logits = run_tflite_inference(interpreter, img_np)
    # Output may be [1, 6, 12, 12] or [1, 12, 12, 6] depending on converter
    if tfl_logits.shape[-1] == NUM_CLASSES:  # NHWC format
        tfl_pred = tfl_logits[0].argmax(axis=-1)
    else:  # NCHW format
        tfl_pred = tfl_logits[0].argmax(axis=0)
    
    match_count += (pt_pred == tfl_pred).sum()
    total_cells += pt_pred.size

agreement = match_count / total_cells
print(f'PyTorch ↔ TFLite agreement: {agreement:.1%}')
print('(>95% is good — small differences expected from int8 quantization)')

## 10. Convert TFLite to C Header for ESP32

The final step: convert the `.tflite` file to a C byte array that
can be included in the ESP32 firmware.

This generates `fomo_model.h` which the Rust code includes via `include_bytes!()`.

In [ ]:
# ── Generate the model as a raw .tflite binary ───────────────────────────
# The Rust side will use include_bytes!() directly on the .tflite file.
# We also generate a C header as a fallback for C/C++ builds.

import shutil

# Copy the tflite model to the esp32 project directory
esp_model_dir = Path('../esp32-fomo/model')
esp_model_dir.mkdir(parents=True, exist_ok=True)

dest = esp_model_dir / 'fomo_model_int8.tflite'
shutil.copy2(tflite_path, dest)
print(f'Model copied to: {dest}')

# Also generate a C header (optional, for C/C++ interop)
with open(tflite_path, 'rb') as f:
    model_bytes = f.read()

header_path = esp_model_dir / 'fomo_model.h'
with open(header_path, 'w') as f:
    f.write('// Auto-generated — do not edit\n')
    f.write(f'// Model size: {len(model_bytes)} bytes\n')
    f.write(f'// Input:  [1, 3, {INPUT_SIZE}, {INPUT_SIZE}] int8\n')
    f.write(f'// Output: [1, {NUM_CLASSES}, {GRID_SIZE}, {GRID_SIZE}] int8\n')
    f.write(f'// Classes: {FOMO_CLASS_NAMES}\n\n')
    f.write('#ifndef FOMO_MODEL_H\n')
    f.write('#define FOMO_MODEL_H\n\n')
    f.write(f'const unsigned int fomo_model_len = {len(model_bytes)};\n')
    f.write('alignas(16) const unsigned char fomo_model_data[] = {\n')
    
    for i in range(0, len(model_bytes), 12):
        chunk = model_bytes[i:i+12]
        hex_str = ', '.join(f'0x{b:02x}' for b in chunk)
        f.write(f'  {hex_str},\n')
    
    f.write('};\n\n')
    f.write('#endif // FOMO_MODEL_H\n')

print(f'C header saved to: {header_path}')
print(f'\nModel size: {len(model_bytes):,} bytes ({len(model_bytes)/1024:.1f} KB)')

# ── Save metadata for Rust side ──────────────────────────────────────────
metadata = {
    'input_size': INPUT_SIZE,
    'grid_size': GRID_SIZE,
    'num_classes': NUM_CLASSES,
    'class_names': FOMO_CLASS_NAMES,
    'normalization': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225],
    },
    'model_size_bytes': len(model_bytes),
}
with open(esp_model_dir / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('\n✓ All export artifacts ready for ESP32!')
print(f'  {dest}')
print(f'  {header_path}')
print(f'  {esp_model_dir / "model_metadata.json"}')